# Model Evaluation

This notebook evaluates the two models defined in `models.py` on the credit card fraud dataset.

## Sections
1. [Data loading & stratified split](#1.-Data-loading-&-stratified-split)
2. [Logistic Regression evaluation](#2.-Logistic-Regression-evaluation)
3. [Random Forest evaluation](#3.-Random-Forest-evaluation)
4. [Model comparison](#4.-Model-comparison)

Because fraud is rare (~1.5% of records), accuracy is misleading and we focus on **recall**, **precision**, **F2** (recall weighted twice as much as precision) and **AUC-PR**, which is the most informative metric under heavy class imbalance.

## 1. Data loading & stratified split

We split the dataset into **80% training** and **20% test** using `stratify=y`. Stratification is essential here: with only ~1.5% positive cases, a random split could easily under-represent fraud in the test set and make the evaluation unreliable. Stratifying preserves the same fraud proportion in both subsets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    fbeta_score,
    precision_recall_curve,
    roc_curve,
)

from models import build_logistic_regression, build_random_forest

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

In [ ]:
df = pd.read_csv("credit_card_fraud_10k.csv")

TARGET = "is_fraud"
X = df.drop(columns=[TARGET, "transaction_id"])
y = df[TARGET]

X = pd.get_dummies(X, drop_first=True)

print(f"Total rows: {len(df)}")
print(f"Fraud share: {y.mean():.4f}  ({int(y.sum())} positives out of {len(y)})")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

split_summary = pd.DataFrame({
    "size":        [len(y_train), len(y_test)],
    "frauds":      [int(y_train.sum()), int(y_test.sum())],
    "fraud_rate":  [y_train.mean(), y_test.mean()],
}, index=["train", "test"])
split_summary

The fraud rate in train and test is virtually identical, confirming that the stratification preserved the original class balance.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

We define a single helper that returns the metrics we care about for fraud detection. Using the same function for both models guarantees the comparison is apples-to-apples.

In [ ]:
import json
from pathlib import Path

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


def evaluate_model(name, y_true, y_pred, y_proba):
    metrics = {
        "model":     name,
        "precision": precision_score(y_true, y_pred),
        "recall":    recall_score(y_true, y_pred),
        "f2":        fbeta_score(y_true, y_pred, beta=2),
        "roc_auc":   roc_auc_score(y_true, y_proba),
        "auc_pr":    average_precision_score(y_true, y_proba),
    }
    print(f"================ {name} ================")
    print("Confusion Matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=4))
    print(f"ROC AUC  : {metrics['roc_auc']:.4f}")
    print(f"AUC-PR   : {metrics['auc_pr']:.4f}")
    print(f"F2 score : {metrics['f2']:.4f}")
    return metrics


def save_results(name, y_true, y_pred, y_proba, metrics):
    slug = name.lower().replace(" ", "_")

    with open(RESULTS_DIR / f"{slug}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    with open(RESULTS_DIR / f"{slug}_classification_report.txt", "w") as f:
        f.write(classification_report(y_true, y_pred, digits=4))

    pd.DataFrame({
        "y_true":  y_true.values,
        "y_pred":  y_pred,
        "y_proba": y_proba,
    }).to_csv(RESULTS_DIR / f"{slug}_predictions.csv", index=False)

    pd.DataFrame(
        confusion_matrix(y_true, y_pred),
        index=["true_legit", "true_fraud"],
        columns=["pred_legit", "pred_fraud"],
    ).to_csv(RESULTS_DIR / f"{slug}_confusion_matrix.csv")

    print(f"Saved results for '{name}' in: {RESULTS_DIR.resolve()}")


def plot_confusion(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["legit", "fraud"],
                yticklabels=["legit", "fraud"], ax=ax, cbar=False)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

## 2. Logistic Regression evaluation

Same configuration as in `models.py`: `class_weight="balanced"` to compensate the imbalance, `max_iter=1000`, fitted on the scaled training set.

In [ ]:
lr_model = build_logistic_regression()
lr_model.fit(X_train_scaled, y_train)

lr_pred  = lr_model.predict(X_test_scaled)
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_metrics = evaluate_model("LOGISTIC REGRESSION", y_test, lr_pred, lr_proba)
save_results("LOGISTIC REGRESSION", y_test, lr_pred, lr_proba, lr_metrics)

Logistic regression with `class_weight="balanced"` re-weights the loss so that the rare fraud class is treated as if it were as frequent as the majority. The expected effect is a sharp increase in recall (fraud caught) at the cost of precision (more false positives).

## 3. Random Forest evaluation

Same hyperparameters as in `models.py`. No scaling is needed for tree-based models. Predictions are taken at the default 0.5 threshold.

In [ ]:
rf_model = build_random_forest()
rf_model.fit(X_train, y_train)

rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_pred  = (rf_proba > 0.5).astype(int)

rf_metrics = evaluate_model("RANDOM FOREST", y_test, rf_pred, rf_proba)
save_results("RANDOM FOREST", y_test, rf_pred, rf_proba, rf_metrics)

Compared to logistic regression, the random forest can capture non-linear interactions between features (for example, the joint effect of `foreign_transaction` and `location_mismatch` highlighted in the EDA) without manual feature engineering.

## 4. Model comparison

We compare the two models on the metrics that matter for fraud detection.

In [ ]:
comparison = pd.DataFrame([lr_metrics, rf_metrics]).set_index("model")
comparison.to_csv(RESULTS_DIR / "comparison.csv")
comparison.style.format("{:.4f}").background_gradient(cmap="Greens")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_confusion(y_test, lr_pred, "Logistic Regression", axes[0])
plot_confusion(y_test, rf_pred, "Random Forest",       axes[1])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for name, proba, color in [
    ("Logistic Regression", lr_proba, "#4C78A8"),
    ("Random Forest",       rf_proba, "#E45756"),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})", color=color)

    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(rec, prec, label=f"{name} (AP={average_precision_score(y_test, proba):.3f})", color=color)

axes[0].plot([0, 1], [0, 1], "k--", linewidth=0.8)
axes[0].set_title("ROC curve")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend(loc="lower right")

axes[1].set_title("Precision-Recall curve")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="lower left")

plt.tight_layout()
plt.show()

### How to read the comparison

- **Recall** measures the share of actual frauds that are caught. In a fraud-detection setting missing a fraud is typically much more costly than investigating a false alarm, so this is usually the most important metric.
- **Precision** measures how trustworthy a positive prediction is — high precision means few false alarms.
- **F2** combines the two but weighs recall twice as much as precision, reflecting the asymmetric cost above.
- **AUC-PR** (average precision) is the most informative *threshold-independent* metric under strong class imbalance: ROC-AUC tends to look optimistic because the negative class dominates, while AUC-PR focuses on the minority class.

**Expected pattern.** Logistic regression with balanced class weights typically achieves very high recall but low precision (many false positives). Random forest, being more flexible, usually delivers a better precision-recall trade-off and a higher AUC-PR, which makes it the stronger candidate for a production-like setting. The exact numbers from this run are reported in the table and curves above.

## Results and model comparison

The two models display very different behaviours on the test set, and the comparison clearly highlights the classic precision/recall trade-off that arises in an imbalanced problem like fraud detection.

**Logistic Regression**
- **Recall = 1.000** → it catches *all* the frauds in the test set: zero false negatives.
- **Precision = 0.266** → however, only about 1 alert in 4 is an actual fraud; the other 3 are false positives.
- **AUC-PR = 0.738** → acceptable but clearly lower than the alternative.
- **F2 = 0.644** → penalised by the low precision, even though F2 already weighs recall twice as much.

In practice, logistic regression with `class_weight="balanced"` casts a wide net: in order not to miss any fraud, it flags many legitimate transactions as suspicious. This behaviour can be useful when the cost of a false negative is far greater than that of a false positive (e.g. cheap manual review), but operationally it generates a lot of noise.

**Random Forest**
- **Precision = 1.000** → every alert raised is a real fraud, zero false positives.
- **Recall = 0.800** → it catches 80% of frauds and misses the remaining 20%.
- **AUC-PR = 0.998** → essentially perfect, and this is the most reliable metric under strong class imbalance.
- **ROC-AUC = 1.000** and **F2 = 0.833** → both better than logistic regression.

The Random Forest exploits non-linear interactions between features (e.g. the `foreign_transaction × location_mismatch` combination that emerged in the EDA) and produces a much more "selective" classifier: very reliable on the positives it predicts, even if at the default 0.5 threshold it cannot cover the last 20% of frauds.

**Verdict**

The Random Forest is overall the stronger model: AUC-PR close to 1, higher F2, perfect ROC-AUC. The fact that logistic regression wins on raw recall is an artefact of the aggressiveness induced by `class_weight="balanced"`, not a sign of superiority — in production the same logistic regression would pay for it with an unsustainable volume of false alarms.

A natural next step would be to **lower the Random Forest decision threshold** below 0.5 (e.g. by picking an operating point on the precision-recall curve) to recover part of that missing 20% recall without losing too much precision. With an AUC-PR of 0.998 there is plenty of room to do so.